# Extending Opaque

This notebook walks through the contributor surface in
`opaque.api.*` — the same plug-in points the built-in mechanisms use.
It is the only tutorial that imports from `opaque.api.*`; user-facing
code lives behind public façades like `opaque.dpsgd.*` and
`opaque.dpftrl.*`.

**Companion guides:**
[Extending Opaque overview](../extending/index.md),
[Serialization registry](../extending/serialization.md),
[Distributed sync registry](../extending/distributed-sync.md),
[Clipping `fun` helpers](../extending/clipping-fun.md).

**You will:**

1. Register a custom serializer for a new state class.
2. Register a sync handler for the same state under DDP.
3. Use `clipped_fun` (the lower-level clipping helper) to build a
   custom mechanism that doesn't fit the standard `clipped_grad`
   shape.

## 1. Register a custom serializer

Opaque's checkpointing pipeline is a type-keyed registry. Anything
registered round-trips through the public `state_dict` /
`from_state_dict` API. Side-effect-import the module that calls
`register_serializer` and your type is checkpointable.

In [1]:
from dataclasses import dataclass
from typing import Any, Mapping

from opaque.api.base.serialization import register_serializer
from opaque.serialization import state_dict, from_state_dict


@dataclass
class TrimmedGaussianState:
    """Custom DP state: Gaussian truncated to a Mahalanobis ball."""
    noise_multiplier: float
    radius: float
    step: int = 0


def _dump_trimmed(obj: TrimmedGaussianState) -> dict[str, Any]:
    return {
        "type": "TrimmedGaussianState/1",
        "noise_multiplier": obj.noise_multiplier,
        "radius": obj.radius,
        "step": obj.step,
    }


def _load_trimmed(_template, sd: Mapping[str, Any]) -> TrimmedGaussianState:
    assert sd["type"] == "TrimmedGaussianState/1", sd["type"]
    return TrimmedGaussianState(
        noise_multiplier=sd["noise_multiplier"],
        radius=sd["radius"],
        step=sd["step"],
    )


register_serializer(TrimmedGaussianState, _dump_trimmed, _load_trimmed)

# Round-trip through the public API.
obj = TrimmedGaussianState(noise_multiplier=1.5, radius=2.0, step=42)
blob = state_dict(obj)
print("Serialized:", blob)

restored = from_state_dict(
    TrimmedGaussianState(noise_multiplier=0.0, radius=0.0), blob,
)
print("Restored:  ", restored)
assert restored == obj

Serialized: {'type': 'TrimmedGaussianState/1', 'noise_multiplier': 1.5, 'radius': 2.0, 'step': 42}
Restored:   TrimmedGaussianState(noise_multiplier=1.5, radius=2.0, step=42)


Tag dumps with a version string (`"TrimmedGaussianState/1"`) so future
schema changes can dispatch through a migration path.

## 2. Register a sync handler for DDP

`opaque.distributed.sync(state)` dispatches by exact type. Register a
handler so cross-rank reductions happen automatically when your state
is used inside a DDP run. Outside DDP
(`is_distributed() is False`), the handler should pass through.

In [2]:
from dataclasses import replace

from opaque.api.engine.distributed import register_sync_type
from opaque.distributed import is_distributed, sync


def _sync_trimmed(state: TrimmedGaussianState) -> TrimmedGaussianState:
    if not is_distributed():
        return state
    # In a real DDP setting, this would all_reduce the radius across ranks.
    # Here we just demonstrate the registration shape.
    return replace(state, radius=state.radius)


register_sync_type(TrimmedGaussianState, _sync_trimmed)

# Outside DDP, sync() returns the state unchanged.
synced = sync(obj)
print("Synced (single rank, no-op):", synced)
assert synced == obj

Synced (single rank, no-op): TrimmedGaussianState(noise_multiplier=1.5, radius=2.0, step=42)


## 3. Build a custom mechanism with `clipped_fun`

Most DP code uses `opaque.dpsgd.clipping.clipped_grad`, which builds
on `vmap(grad(...))`. When you need to clip a non-loss-gradient
quantity — a per-example embedding, a privacy-bounded statistic, an
activation summary — drop down to `clipped_fun`.

Below: clip per-example dot-product features, add Gaussian noise,
and produce a privately released summary statistic.

In [3]:
import torch

from opaque.api.engine.clipping.fun import clipped_fun
from opaque.dpsgd.noise import gaussian_noise
from opaque.random import key


def per_example_summary(params: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Per-example summary statistic — a single scalar per record."""
    return torch.dot(x, params)


params = torch.randn(10)
batch = torch.randn(64, 10)

clipped_fn, clip_state = clipped_fun(
    per_example_summary,
    batch_argnums=1,
    clipping_norm=1.0,
    normalize_by=batch.shape[0],
)
clipped, clip_state = clipped_fn(params, batch, state=clip_state)

noise_fn, noise_state = gaussian_noise(noise_multiplier=1.0, key=key(0))
noised, noise_state = noise_fn(clipped, noise_state)

print(f"Clipped pytree: {type(clipped).__name__}, max_norm={clipped.max_norm}")
print(f"Noised pytree:  {type(noised).__name__}")
print(f"Noise stddev:   {noised.noise_stddev:.4f}")
print(f"Released value: {noised.pytree.item():.4f}")

Clipped pytree: ClippedPytree, max_norm=0.015625
Noised pytree:  NoisedPytree
Noise stddev:   0.0156
Released value: -0.1685


The clipped output flows into `gaussian_noise` exactly like a
DP-SGD gradient does — sensitivity is read off the `ClippedPytree`,
noise is calibrated to that bound. The same path works for any
per-example quantity you can compute.

Lower still: `clip_pytree` accepts an already-batched pytree and
skips both `vmap` and `grad`. Use it when you've produced per-example
tensors via some other route (custom autograd, manual finite
differencing, gradient checkpointing).

## Where this fits

These three primitives — serializer registry, sync registry, and the
`fun`/`pytree` clipping helpers — are the building blocks for a new
mechanism family:

1. Define your state classes; register serializers so checkpoints
   round-trip.
2. Register sync handlers for cross-rank reductions in DDP.
3. Build the mechanism on top of the existing clipping helpers (or
   write a new clipping rule entirely if your sensitivity bound
   needs it).
4. Expose a public façade under `opaque.<concern>` or
   `opaque.<stack>.<concern>` that re-exports the impl.

The full contributor walkthrough — including how to add an accounting
factory and where the layering lines run — lives in
[Adding a new mechanism family](../extending/new-mechanism.md).